# 09 — Swaptions

- **Black swaption pricing** (log-normal)
- **Bachelier swaption pricing** (normal)
- **Hull-White swaption** (Jamshidian decomposition)
- **Payer vs receiver parity**
- **AD Greeks**: dV/dVol, dV/dRate

In [ ]:
BACKEND = "cpu"

import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), "..") if not os.getcwd().endswith("ql-jax") else ".")
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath("__file__")), ".."))

from notebooks._common import setup_backend, load_quantlib, compare_table, timed_ms
jax = setup_backend(BACKEND)
import jax.numpy as jnp
import numpy as np
QL = load_quantlib()

In [ ]:
from ql_jax.instruments.swaption import make_swaption
from ql_jax.engines.swaption.black import black_swaption_price, bachelier_swaption_price, par_swap_rate
from ql_jax.engines.swaption.hull_white import hw_swaption_price

FLAT_RATE = 0.05
EXERCISE = 5.0        # 5Y option
SWAP_TENOR = 5.0      # into 5Y swap
FIXED_RATE = 0.05     # ATM
NOTIONAL = 1_000_000.0
BLACK_VOL = 0.20      # log-normal vol
NORMAL_VOL = 0.01     # normal vol
HW_A = 0.1            # HW mean reversion
HW_SIGMA = 0.01       # HW vol

def flat_disc(t):
    return jnp.exp(-FLAT_RATE * t)

---
## 1. Black Swaption (Log-Normal)

In [ ]:
# Create swaption
swpn_pay = make_swaption(
    exercise_time=EXERCISE,
    swap_tenor=SWAP_TENOR,
    fixed_rate=FIXED_RATE,
    frequency=0.5,
    notional=NOTIONAL,
    payer=True)

swpn_rec = make_swaption(
    exercise_time=EXERCISE,
    swap_tenor=SWAP_TENOR,
    fixed_rate=FIXED_RATE,
    frequency=0.5,
    notional=NOTIONAL,
    payer=False)

# Forward par swap rate
fwd_rate = float(par_swap_rate(
    flat_disc, EXERCISE, swpn_pay.swap_payment_dates, swpn_pay.swap_accrual_fractions))
print(f"Forward par swap rate: {fwd_rate:.6f}  (should ≈ {FLAT_RATE})")

# Black prices
payer_black = float(black_swaption_price(swpn_pay, flat_disc, fwd_rate, BLACK_VOL))
recvr_black = float(black_swaption_price(swpn_rec, flat_disc, fwd_rate, BLACK_VOL))

print(f"Payer swaption (Black) : {payer_black:12.2f}")
print(f"Receiver swaption      : {recvr_black:12.2f}")
print(f"Payer − Receiver       : {payer_black - recvr_black:12.2f}  (should ≈ swap NPV ≈ 0)")

---
## 2. Bachelier Swaption (Normal Model)

In [ ]:
payer_bach = float(bachelier_swaption_price(swpn_pay, flat_disc, fwd_rate, NORMAL_VOL))
recvr_bach = float(bachelier_swaption_price(swpn_rec, flat_disc, fwd_rate, NORMAL_VOL))

print(f"Payer (Bachelier)  : {payer_bach:12.2f}")
print(f"Receiver           : {recvr_bach:12.2f}")

---
## 3. Hull-White Swaption (Jamshidian Decomposition)

In [ ]:
payer_hw = float(hw_swaption_price(
    NOTIONAL, FIXED_RATE,
    swpn_pay.swap_payment_dates, swpn_pay.swap_accrual_fractions,
    HW_A, HW_SIGMA, flat_disc, EXERCISE, is_payer=True))

recvr_hw = float(hw_swaption_price(
    NOTIONAL, FIXED_RATE,
    swpn_rec.swap_payment_dates, swpn_rec.swap_accrual_fractions,
    HW_A, HW_SIGMA, flat_disc, EXERCISE, is_payer=False))

print(f"Payer (HW)    : {payer_hw:12.2f}")
print(f"Receiver (HW) : {recvr_hw:12.2f}")

In [ ]:
# Summary table
print(f"\n{'Model':<20s} {'Payer':>12s} {'Receiver':>12s}")
print("-" * 50)
print(f"{'Black':<20s} {payer_black:>12.2f} {recvr_black:>12.2f}")
print(f"{'Bachelier':<20s} {payer_bach:>12.2f} {recvr_bach:>12.2f}")
print(f"{'Hull-White':<20s} {payer_hw:>12.2f} {recvr_hw:>12.2f}")

---
## 4. Swaption Vol Surface

Price across a grid of expiries and underlying tenors.

In [ ]:
import matplotlib.pyplot as plt

expiries = [1, 2, 3, 5, 7, 10]
tenors   = [1, 2, 3, 5, 7, 10]

price_matrix = np.zeros((len(expiries), len(tenors)))
for i, exp in enumerate(expiries):
    for j, tn in enumerate(tenors):
        sw = make_swaption(float(exp), float(tn), FIXED_RATE, 0.5, NOTIONAL, True)
        fwd = float(par_swap_rate(flat_disc, float(exp),
                                   sw.swap_payment_dates, sw.swap_accrual_fractions))
        price_matrix[i, j] = float(black_swaption_price(sw, flat_disc, fwd, BLACK_VOL))

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(price_matrix / 1000, aspect='auto', origin='lower', cmap='YlOrRd')
ax.set_xticks(range(len(tenors)))
ax.set_xticklabels([f"{t}Y" for t in tenors])
ax.set_yticks(range(len(expiries)))
ax.set_yticklabels([f"{e}Y" for e in expiries])
ax.set_xlabel('Underlying Tenor')
ax.set_ylabel('Option Expiry')
ax.set_title('Swaption Prices (×1000, Black 20% vol)')
for i in range(len(expiries)):
    for j in range(len(tenors)):
        ax.text(j, i, f"{price_matrix[i,j]/1000:.0f}k", ha='center', va='center', fontsize=8)
fig.colorbar(im)
plt.show()

---
## 5. AD: Swaption Vega

In [ ]:
def swaption_price_fn(vol):
    sw = make_swaption(EXERCISE, SWAP_TENOR, FIXED_RATE, 0.5, NOTIONAL, True)
    fwd = par_swap_rate(flat_disc, EXERCISE, sw.swap_payment_dates, sw.swap_accrual_fractions)
    return black_swaption_price(sw, flat_disc, fwd, vol)

vols = jnp.linspace(0.05, 0.50, 50)
prices_vs_vol = [float(swaption_price_fn(v)) for v in vols]
vegas_vs_vol = [float(jax.grad(swaption_price_fn)(v)) for v in vols]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(np.array(vols)*100, np.array(prices_vs_vol)/1000, 'b-', linewidth=2)
ax1.set_xlabel('Vol (%)')
ax1.set_ylabel('Price (×1000)')
ax1.set_title('5Y×5Y Swaption Price vs Vol')
ax1.grid(True, alpha=0.3)

ax2.plot(np.array(vols)*100, np.array(vegas_vs_vol)/1000, 'r-', linewidth=2)
ax2.set_xlabel('Vol (%)')
ax2.set_ylabel('Vega (×1000)')
ax2.set_title('Swaption Vega via AD')
ax2.grid(True, alpha=0.3)

fig.tight_layout()
plt.show()

---
## 6. Exercises

1. **Payer-receiver parity**: Verify that payer − receiver = swap NPV for an off-market strike.
2. **HW calibration**: Given market swaption prices at 5 expiries, calibrate $a$ and $\sigma$ using `jax.grad`-based optimization.
3. **Gamma**: Compute $\partial^2 V / \partial S_{\text{fwd}}^2$ for the swaption.

---
*End of Notebook 09*